# Getting started

`GaugeFixer` facilitates interpretation of sequence-function models by removing unconstrained degrees of freedom from the parameter values of one-hot generalized models, such as additive, pairwise and all-order interaction models, and express them in ways that have different interpretations that may be useful for different applications.


> **Note**: The parameters of these models (or the sequence-function map they encode) must be learned before defining a model in `GaugeFixer`.

In this section, we show a simple example for how to define an all-order interaction model and compute the corresponding gauge-fixed coefficients under different gauges using `GaugeFixer`'s efficient algorithms.
For more details on `GaugeFixer` see [User guide](tutorial.ipynb) and for a real world application see [Example](example.ipynb).

### Importing required libraries


In [1]:
from gaugefixer import AllOrderModel

### Defining a sequence-function model

`GaugeFixer` considers sequence-function models for sequences of fixed length `L` with characters drawn from an alphabet. This alphabet can be common to all positions or site specific. However, we will typically consider biological sequences defined over `'dna'`, `'rna` or `'protein'` alphabets. 

For this example, we will define model for 9-nucleotide long RNA sequences for the Shine-Dalgarno sequence landscape inferred by [Martí-Gómez et al. (2026)](https://academic.oup.com/mbe/article/doi/10.1093/molbev/msag023/8456298) from data by [Kuo et al. (2020)](https://genome.cshlp.org/content/30/5/711) as follows:

In [2]:
model = AllOrderModel(L=9, alphabet_name='rna')
model

AllOrderModel(L=9,alphabet_name=rna,n_features=1953125,n_orbits=512)

There are multiple ways of defining the parameters of the model, but the most direct way is by providing the values of the parameters associated to the one-hot features as a `pd.Series` using the `set_params` method. The series must be indexed by the sequence binary features $x_U^u$, each of which must be defined as a tuple containing the set of sites $U$ and the subsequence $u$. Here, we load the set of parameters from a file

> **Note**: sites are 0-indexed in `GaugeFixer`

In [3]:
from pickle import load
with open('shine_dalgarno.theta.pkl', 'rb') as fhand:
    theta = load(fhand)
theta

((), )                                      0.620094
((0,), A)                                   0.031795
((0,), C)                                  -0.105760
((0,), G)                                   0.104421
((0,), U)                                  -0.030456
                                              ...   
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUGU)    0.005219
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUA)   -0.001907
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUC)    0.000431
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUG)    0.001947
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUU)   -0.000471
Length: 1953125, dtype: float64

and use them to set the parameters of the sequence-function model previously defined by simply using `set_params`

In [4]:
model.set_params(theta)


### Fixing the gauge

Now that the model is completely specified, we may want to interpret the parameter values. However, there are subspaces of parameter values that encode the same sequence-to-function model. Fixing the gauge means choosing one among the many set of parameter values that encode the same model by specifying certain properties of the parameter values e.g. in the `hierarchical` gauge the parameters associated to low order subsequences explain as much variance as possible.

We can do this operation easily by using the method `get_fixed_params`. The gauge is defined either by their specific names, e.g., `wild-type`, `zero-sum`, or `hierarchical`, or by directly defining the $\lambda$ value as `lda` and the site- and allele-specific probabilities `pi_lc`, which define a specific gauge in the more general $\lambda-\pi$ family of linear gauges (See [User guide](tutorial.ipynb) for more details).

For instance, lets express our parameters in the commonly used `zero-sum` gauge.


In [5]:
theta_fixed = model.get_fixed_params(gauge='zero-sum')
theta_fixed

((), )                                      0.620094
((0,), A)                                   0.031795
((0,), C)                                  -0.105760
((0,), G)                                   0.104421
((0,), U)                                  -0.030456
                                              ...   
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUGU)    0.005219
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUA)   -0.001907
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUC)    0.000431
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUG)    0.001947
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUU)   -0.000471
Length: 1953125, dtype: float64

### Interpreting parameter values

Now the parameters can be more easily interpreted according to the `zero-sum` gauge, where the parameters represent the average effect of placing subsequence $u$ at positions $U$ relative to the lower order expectations. For example, lower order terms can be interpreted as follows under the `zero-sum` gauge:

- `((), '')` represents the empty set and the associated parameter represents the average phenotype across all possible sequences.
- `((i), c)` represents the average effect of placing allele $c$ at position $i$ across all possible sequences.
- `((i,j), c_1c_2)` represents the average effect of placing alleles $c_1$ and $c_2$ at positions $i,j$ across all possible sequences relative to the additive expectation.

In this particular case, we can see the average phenotype given by


In [6]:
theta_fixed.loc[[((), '')]]

((), )    0.620094
dtype: float64

the average effect of placing `'A'` at position `1`

In [7]:
theta_fixed.loc[[((1,), 'A')]]

((1,), A)    0.054673
dtype: float64

the average effect of placing `'G'` at position `2`

In [8]:
theta_fixed.loc[[((2,), 'G')]]

((2,), G)    0.134858
dtype: float64

as well as the additional effect of having the two alleles together relative to the expectation from the single allelic effects

In [9]:
theta_fixed.loc[[((1, 2), "AG")]]


((1, 2), AG)    0.085525
dtype: float64

Likewise, we can also interprete the values of the higher-order terms relative to the lower order expectation. For instance, what is the additional effect of placing subsequence `u=GAG` at positions `U=(0, 1, 2)` relative to the pairwise model expectation?

In [10]:
theta_fixed.loc[[((0, 1, 2), "GAG")]]

((0, 1, 2), GAG)    0.231421
dtype: float64

In this section, we have shown how `GaugeFixer` can be used for removing unconstrained degrees of freedom from parameter values and obtaining gauge-fixed parameters with a specific interpretation given by the `zero-sum` gauge. 
However, rather than interpreting parameters expressed in a single gauge, different gauges are typically useful for different applications. For instance, in the [Example](example.ipynb) section, we use different hierarchical gauges to understand whether and how the effects of mutations at the AGGAG core motif of the Shine-Dalgarno sequence changes when it is placed at different distances relative to the start codon described in the `GaugeFixer` manuscript. 